# NutriWise AI — walkthrough notebook

This notebook exercises each stage of the pipeline on its own, then the full
pipeline, storage, and finally launches the Gradio app inline.

Run it with the project's virtual environment as the kernel, from the `notebooks/` folder.

In [ ]:
import sys, logging
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # make the `app` package importable from notebooks/
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(name)s: %(message)s")
for noisy in ("httpx", "httpx2", "google_genai"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

## 1. Parser and local food table (Stage 1)

In [ ]:
from app.food_table import parse_meal, estimate_from_table

for meal in ["2 rotis and a bowl of dal", "half a plate of poha, 2 cups of chai", "masala dosa and coffee"]:
    print(meal, "->", [(i.quantity, i.unit, i.name) for i in parse_meal(meal)])
    print("   ", estimate_from_table(meal))

## 2. Nutrition APIs (Stage 2)

In [ ]:
from app.apis import estimate_from_apis

estimate_from_apis("200g grilled chicken breast")

## 3. LLM fallback chain (Stage 3)

In [ ]:
from app.llm import estimate_with_llm

estimate_with_llm("a bowl of grandma's mystery stew")

## 4. The full pipeline — which stage answers each meal?

In [ ]:
import pandas as pd
from app.estimator import estimate_meal

meals = [
    "2 rotis and a bowl of dal",
    "3 idlis with sambar",
    "200g grilled chicken breast and a bowl of rice",
    "2 roti and a slice of pizza",
    "1 plate pav bhaji with extra butter",
    "a bowl of grandma's mystery stew",
]
pd.DataFrame([{"input": m, **estimate_meal(m).model_dump()} for m in meals])[
    ["input", "source", "calories", "protein", "carbs", "fat", "note"]
]

## 5. Storage — save, total, delete (uses a temporary file, not your real log)

In [ ]:
import tempfile
from app import storage

storage.DATA_FILE = Path(tempfile.mkdtemp()) / "meals.json"
a = storage.save_meal(estimate_meal("2 rotis and dal"), "2 rotis and dal")
b = storage.save_meal(estimate_meal("3 idlis with sambar"), "3 idlis with sambar")
print(storage.daily_totals())
storage.delete_meal(a["id"])
print(storage.daily_totals())

## 6. Launch the app inline

Restart the kernel first if you ran section 5, so the app uses your real meal log.

In [ ]:
from app.ui import launch

launch()